In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install --upgrade gspread gspread-dataframe
from google.colab import auth
auth.authenticate_user()

import gspread
from gspread_dataframe import get_as_dataframe

import pandas as pd
import requests
import datetime
import csv
import google.generativeai as genai
from google.colab import files
import pandas as pd
import gspread
from google.auth import default
from gspread_dataframe import get_as_dataframe

In [ ]:
# Gemini API
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel("models/gemini-1.5-pro-latest")


# Load match info
df = pd.read_csv("/content/drive/MyDrive/CricTech/CricTech_IPL_General_Info.csv")

# Load all player stats
all_players = pd.read_csv("/content/drive/MyDrive/CricTech/CricTech_Player_Old_Stats.csv")
all_players["Player Name"] = all_players["Player Name"].str.strip().str.lower()
all_players["Team"] = all_players["Team"].str.strip().str.upper()

# Weather config
weather_team_config = {
    "sunny": [5, 2, 2, 2],
    "cloudy": [3, 3, 3, 2],
    "clear sky": [4, 3, 2, 2],
    "rainy": [4, 2, 3, 2],
    "windy": [3, 4, 2, 2],
    "foggy": [2, 4, 3, 2],
    "humid": [3, 3, 3, 2],
    "cold": [3, 4, 2, 2],
    "autumn": [3, 3, 3, 2],
    "overcast": [2, 4, 3, 2],
    "dewy": [4, 2, 3, 2],
    "unknown": [3, 4, 2, 2]
}

def classify_weather(description):
    description = description.lower()
    for key in weather_team_config.keys():
        if key in description:
            return key, weather_team_config[key]
    return "unknown", weather_team_config["unknown"]

pitch_team_config = {
    "batting-friendly": [5, 2, 2, 2],
    "spin-friendly": [3, 3, 3, 2],
    "balanced": [4, 3, 2, 2],
    "bowler-friendly": [3, 4, 2, 2],
    "slow, spin-friendly": [3, 3, 3, 2],
    "seam-friendly": [2, 4, 3, 2],
    "new venue": [3, 3, 3, 2]
}

def classify_pitch(pitch_type):
    return pitch_team_config.get(pitch_type.lower(), [3, 4, 2, 2])

def merge_team_configs(weather_config, pitch_config):
    combined = [(w + p) // 2 for w, p in zip(weather_config, pitch_config)]
    diff = 11 - sum(combined)
    combined[0] += diff
    return combined

def get_forecast(lat, lon, match_datetime):
    API_KEY = API_KEY
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
    response = requests.get(url)
    data = response.json()
    if response.status_code == 200 and "list" in data:
        forecast_data = data["list"]
        best_match = min(forecast_data, key=lambda x: abs(datetime.datetime.strptime(x["dt_txt"], "%Y-%m-%d %H:%M:%S") - match_datetime))
        description = best_match["weather"][0]["description"]
        _, weather_config = classify_weather(description)
        return weather_config
    return weather_team_config["unknown"]

SF = {'CHE':'Chennai Super Kings', 'DC':'Delhi Capitals', 'GT':'Gujarat Titans', 'KKR':'Kolkata Knight Riders', 'LSG':'Lucknow Super Giants', 'MI':'Mumbai Indians', 'PBKS':'Punjab Kings', 'RCB':'Royal Challengers Bengaluru', 'RR':'Rajasthan Royals', 'SRH':'Sunrisers Hyderabad'}

def get_key_from_value(input_dict, value):
    for key, val in input_dict.items():
        if val == value:
            return key
    return None



def get_dream11_team(player_data, required, df_original):
    prompt_main = f"""
            You are a professional fantasy cricket expert.
            Select the best Dream11 team from the following players.
            Use previous match performance to assign the best team.
            Only select:

            - {required['BAT']} BAT
            - {required['BOWL']} BOWL
            - {required['ALL']} ALL
            - {required['WK']} WK

            Total should be exactly 11 players. Select also 4 substitutes as:

            - 1 BAT
            - 1 ALL
            - 1 BAT
            - 1 BOWL

            Total credit of main 11 should be <= 100.

            Pick 1 Captain (C) from top player in  BAT, 1 Vice Captain (VC) from top player in ALL, rest are Normal (NA) including substitutes.

            Only respond in the following table format:
            Team,Player Name,Player Type,Credits,LogicScore,C/VC


            Here is the player data:
            {player_data}
            """
    try:
        response = model.generate_content(prompt_main, generation_config={"temperature": 0.7})
        output = response.text.strip()

        lines = output.splitlines()
        rows = []
        for line in lines:
            if not line.strip() or line.lower().startswith("team,"):
                continue
            parts = [x.strip() for x in line.split(",")]
            if len(parts) == 6:
                rows.append([parts[1], parts[0], parts[5]])  # Player Name, Team, C/VC

        with open("CricTech_Output.csv", "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Player Name", "Team", "C/VC"])
            writer.writerows(rows)

        files.download('CricTech_OutputC.csv')

    except Exception as e:
        print("Error generating team:", e)





def run_fantasy_selector():
    match_number = int(input("Enter Match Number: "))

    # Step 1: Match Info
    match = df[df["Match Number"] == match_number]
    if match.empty:
        print("Invalid Match Number!")
        return

    venue = match.iloc[0]["Venue"]
    home_team = match.iloc[0]["Team1"]
    away_team = match.iloc[0]["Team2"]
    lat, lon = match.iloc[0]["Latitude"], match.iloc[0]["Longitude"]
    pitch_type = match.iloc[0]["Pitch Type"]
    match_date = match.iloc[0]["Date"]
    match_time = match.iloc[0]["Time"]
    match_datetime = datetime.datetime.strptime(f"{match_date} {match_time}", "%Y-%m-%d %H:%M")

    # Step 2: Load announced players
    import pandas as pd
    import gspread
    from google.auth import default
    from gspread_dataframe import get_as_dataframe

    creds, _ = default()
    gc = gspread.authorize(creds)

    # Open by name or URL
    sh = gc.open("SquadPlayerNames_IndianT20League")
    worksheet = sh.worksheet(f"Match_{match_number}")

    # Convert to DataFrame
    today_players = get_as_dataframe(worksheet)


    # Convert to DataFrame
    # Convert to DataFrame
    today_players = get_as_dataframe(worksheet)
    today_players["Player Name"] = today_players["Player Name"].str.strip().str.lower()
    today_players["Team"] = today_players["Team"].str.strip().str.upper()
    playing_today = today_players[today_players["IsPlaying"] == "PLAYING"].copy()

    today_players["Player Name"] = today_players["Player Name"].str.strip().str.lower()
    today_players["Team"] = today_players["Team"].str.strip().str.upper()
    playing_today = today_players[today_players["IsPlaying"] == "PLAYING"].copy()
    # Step 3: Merge with player stats
    merged_df = pd.merge(playing_today, all_players)
    merged_df = merged_df.sort_values(by="lineupOrder")
    merged_df["Player Name"] = merged_df["Player Name"].str.title()

    final_df = merged_df[[
        "Player Name", "Team", "Player Type", "Credits", "Average"
    ]]

    # Step 4: Weather & Pitch Logic
    weather_config = get_forecast(lat, lon, match_datetime)
    pitch_config = classify_pitch(pitch_type)
    final_team_config = merge_team_configs(weather_config, pitch_config)

    required = {
        "BAT": final_team_config[0],
        "ALL": final_team_config[1],
        "BOWL": final_team_config[2],
        "WK": final_team_config[3],
    }

    # Step 5: Prepare player data format
    player_data = ""
    for _, row in final_df.iterrows():
        player_data += f"{row['Team']},{row['Player Name']},{row['Player Type']},{row['Credits']},{row['Average']}\n"

    # Step 6: Generate team

    get_dream11_team(player_data, required, final_df)

run_fantasy_selector()


Enter Match Number: 35
Error generating team: Cannot find file: CricTech_OutputC.csv


In [ ]:


# Gemini API
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel("models/gemini-1.5-pro-latest")

# Load match info
df = pd.read_csv("/content/drive/MyDrive/CricTech/CricTech_IPL_General_Info.csv")

# Load all player stats
players_old = pd.read_csv("/content/drive/MyDrive/CricTech/CricTech_Player_Old_Stats.csv")
batters_stats=pd.read_csv("/content/drive/MyDrive/CricTech/CricTech_Bats_2025Stats.csv")
bowlers_stats=pd.read_csv("/content/drive/MyDrive/CricTech/CricTech_Bowlers_2025Stats.csv")

# Weather config
weather_team_config = {
    "sunny": [5, 2, 2, 2],
    "cloudy": [3, 3, 3, 2],
    "clear sky": [4, 3, 2, 2],
    "rainy": [4, 2, 3, 2],
    "windy": [3, 4, 2, 2],
    "foggy": [2, 4, 3, 2],
    "humid": [3, 3, 3, 2],
    "cold": [3, 4, 2, 2],
    "autumn": [3, 3, 3, 2],
    "overcast": [2, 4, 3, 2],
    "dewy": [4, 2, 3, 2],
    "unknown": [3, 4, 2, 2]
}

def classify_weather(description):
    description = description.lower()
    for key in weather_team_config.keys():
        if key in description:
            return key, weather_team_config[key]
    return "unknown", weather_team_config["unknown"]

pitch_team_config = {
    "batting-friendly": [5, 2, 2, 2],
    "spin-friendly": [3, 3, 3, 2],
    "balanced": [4, 3, 2, 2],
    "bowler-friendly": [3, 4, 2, 2],
    "slow, spin-friendly": [3, 3, 3, 2],
    "seam-friendly": [2, 4, 3, 2],
    "new venue": [3, 3, 3, 2]
}

def classify_pitch(pitch_type):
    return pitch_team_config.get(pitch_type.lower(), [3, 4, 2, 2])

def merge_team_configs(weather_config, pitch_config):
    combined = [(w + p) // 2 for w, p in zip(weather_config, pitch_config)]
    diff = 11 - sum(combined)
    combined[0] += diff
    return combined

def get_forecast(lat, lon, match_datetime):
    API_KEY = API_KEY
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
    response = requests.get(url)
    data = response.json()
    if response.status_code == 200 and "list" in data:
        forecast_data = data["list"]
        best_match = min(forecast_data, key=lambda x: abs(datetime.datetime.strptime(x["dt_txt"], "%Y-%m-%d %H:%M:%S") - match_datetime))
        description = best_match["weather"][0]["description"]
        _, weather_config = classify_weather(description)
        return weather_config
    return weather_team_config["unknown"]

SF = {'CHE':'Chennai Super Kings', 'DC':'Delhi Capitals', 'GT':'Gujarat Titans', 'KKR':'Kolkata Knight Riders', 'LSG':'Lucknow Super Giants', 'MI':'Mumbai Indians', 'PBKS':'Punjab Kings', 'RCB':'Royal Challengers Bengaluru', 'RR':'Rajasthan Royals', 'SRH':'Sunrisers Hyderabad'}

def get_key_from_value(input_dict, value):
    for key, val in input_dict.items():
        if val == value:
            return key
    return None



def get_dream11_team(player_data, required, playing_today,players_old,batters_stats,bowlers_stats):
    prompt_main = f"""
            You are a professional fantasy cricket expert.
            Strictly consider all the details below
            Use previous matches performance to assign the best team.

            Only select:

            - {required['BAT']} BAT
            - {required['BOWL']} BOWL
            - {required['ALL']} ALL
            - {required['WK']} WK

            Total should be exactly 11 players. Select also 4 substitutes as:

            - 1 BAT
            - 1 ALL
            - 1 BAT
            - 1 BOWL

            Total credit of main 11 should be <= 100.
            Strictly note all the details below
            Pick 1 Captain (C) from top player in  BAT, 1 Vice Captain (VC) from top player in ALL, rest are Normal (NA) including substitutes.
            dont give any unwanted data in the output
            Only respond in the following table format:
            Player Name,Team,C/VC

            Here is the players who are in team data:
            {player_data}
            Here is the today's players line up data:
            {playing_today}
            Here is the players old data:
            {players_old}
            Here is the batsman 2025 stats data:
            {batters_stats}
            Here is the bowler 205 stats
            {bowlers_stats}
            """
    try:
        response = model.generate_content(prompt_main, generation_config={"temperature": 0.7})
        output = response.text.strip()

        lines = output.splitlines()
        rows = []
        #Ignore the first line
        for line in lines[1:]:
            if not line.strip() or line.lower().startswith("team,"):
                continue
            parts = [x.strip() for x in line.split(",")]
            if len(parts) == 3:
                rows.append([parts[0], parts[1], parts[2]])  # Player Name, Team, C/VC

        with open("CricTech_Output.csv", "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Player Name", "Team", "C/VC"])
            writer.writerows(rows)

        files.download('CricTech_Output.csv')

    except Exception as e:
        print("Error generating team:", e)





def run_fantasy_selector():
    match_number = int(input("Enter Match Number: "))

    # Step 1: Match Info
    match = df[df["Match Number"] == match_number]
    if match.empty:
        print("Invalid Match Number!")
        return

    venue = match.iloc[0]["Venue"]
    home_team = match.iloc[0]["Team1"]
    away_team = match.iloc[0]["Team2"]
    lat, lon = match.iloc[0]["Latitude"], match.iloc[0]["Longitude"]
    pitch_type = match.iloc[0]["Pitch Type"]
    match_date = match.iloc[0]["Date"]
    match_time = match.iloc[0]["Time"]
    match_datetime = datetime.datetime.strptime(f"{match_date} {match_time}", "%Y-%m-%d %H:%M")

    # Step 2: Load announced players
    import pandas as pd
    import gspread
    from google.auth import default
    from gspread_dataframe import get_as_dataframe

    creds, _ = default()
    gc = gspread.authorize(creds)

    # Open by name or URL
    sh = gc.open("SquadPlayerNames_IndianT20League")
    worksheet = sh.worksheet(f"Match_{match_number}")

    # Convert to DataFrame
    today_players = get_as_dataframe(worksheet)


    # Convert to DataFrame
    today_players = get_as_dataframe(worksheet)

    today_players["Player Name"] = today_players["Player Name"].str.strip().str.lower()
    today_players["Team"] = today_players["Team"].str.strip().str.upper()
    playing_today = today_players[today_players["IsPlaying"] == "PLAYING"].copy()


    # Step 4: Weather & Pitch Logic
    weather_config = get_forecast(lat, lon, match_datetime)
    pitch_config = classify_pitch(pitch_type)
    final_team_config = merge_team_configs(weather_config, pitch_config)

    required = {
        "BAT": final_team_config[0],
        "ALL": final_team_config[1],
        "BOWL": final_team_config[2],
        "WK": final_team_config[3],
    }

    # Step 5: Prepare player data format
    player_data = ""
    for _, row in today_players.iterrows():
        player_data += f"{row['Team']},{row['Player Name']},{row['Player Type']},\n"


    get_dream11_team(player_data, required, playing_today,players_old,batters_stats,bowlers_stats)

run_fantasy_selector()


Enter Match Number: 35


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# prompt: lines = output.splitlines()
# here i need to ignore 1st line

def get_dream11_team(player_data, required, playing_today,players_old,batters_stats,bowlers_stats):
    prompt_main = f"""
            You are a professional fantasy cricket expert.
            Strictly consider all the details below
            Use previous matches performance to assign the best team.

            Only select:

            - {required['BAT']} BAT
            - {required['BOWL']} BOWL
            - {required['ALL']} ALL
            - {required['WK']} WK

            Total should be exactly 11 players. Select also 4 substitutes as:

            - 1 BAT
            - 1 ALL
            - 1 BAT
            - 1 BOWL

            Total credit of main 11 should be <= 100.
            Strictly note all the details below
            Pick 1 Captain (C) from top player in  BAT, 1 Vice Captain (VC) from top player in ALL, rest are Normal (NA) including substitutes.
            dont give any unwanted data in the output
            Only respond in the following table format:
            Player Name,Team,C/VC

            Here is the players who are in team data:
            {player_data}
            Here is the today's players line up data:
            {playing_today}
            Here is the players old data:
            {players_old}
            Here is the batsman 2025 stats data:
            {batters_stats}
            Here is the bowler 205 stats
            {bowlers_stats}
            """
    try:
        response = model.generate_content(prompt_main, generation_config={"temperature": 0.7})
        output = response.text.strip()

        lines = output.splitlines()
        rows = []
        #Ignore the first line
        for line in lines[1:]:
            if not line.strip() or line.lower().startswith("team,"):
                continue
            parts = [x.strip() for x in line.split(",")]
            if len(parts) == 3:
                rows.append([parts[0], parts[1], parts[2]])  # Player Name, Team, C/VC

        with open("CricTech_Output.csv", "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Player Name", "Team", "C/VC"])
            writer.writerows(rows)

        files.download('CricTech_Output.csv')

    except Exception as e:
        print("Error generating team:", e)
